In [2]:
import pandas as pd

In [5]:
# Load the dataset
sales = pd.read_csv('../data/sales_train_evaluation.csv')
calendar = pd.read_csv('../data/calendar.csv')
prices = pd.read_csv('../data/sell_prices.csv')

print(sales.shape)
sales.head()

(30490, 1947)


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [7]:
sales.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1',
       'd_2', 'd_3', 'd_4',
       ...
       'd_1932', 'd_1933', 'd_1934', 'd_1935', 'd_1936', 'd_1937', 'd_1938',
       'd_1939', 'd_1940', 'd_1941'],
      dtype='str', length=1947)

In [11]:
print(sales['store_id'].unique())
print(sales['cat_id'].unique())

<StringArray>
['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2',
 'WI_3']
Length: 10, dtype: str
<StringArray>
['HOBBIES', 'HOUSEHOLD', 'FOODS']
Length: 3, dtype: str


In [12]:
# we can see there are 10 different stores! 
# so, since these are big datasets, maybe we can start with one store first..

In [13]:
# Row count per store, across ALL stores to choose the biggest one among them
store_counts = sales['store_id'].value_counts().sort_values(ascending=False)
print(store_counts)

store_id
CA_1    3049
CA_2    3049
CA_3    3049
CA_4    3049
TX_1    3049
TX_2    3049
TX_3    3049
WI_1    3049
WI_2    3049
WI_3    3049
Name: count, dtype: int64


In [14]:
top_store = store_counts.index[0]
print(top_store)

store_df = sales[sales['store_id'] == top_store]
print(store_df.shape)

CA_1
(3049, 1947)


In [15]:
# Grab the data of CA_1 only
ca1 = sales[sales['store_id'] == 'CA_1'].copy()
print(ca1.shape)

# Identify id columns vs day columns
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in ca1.columns if c.startswith('d_')]

print(f"Number of id columns: {len(id_cols)}")
print(f"Number of day columns: {len(day_cols)}")

# Melt to long format
ca1_long = ca1.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

print(ca1_long.shape)
ca1_long.head()

(3049, 1947)
Number of id columns: 6
Number of day columns: 1941
(5918109, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [16]:
# Filter to FOODS category within CA_1
ca1_foods = ca1[ca1['cat_id'] == 'FOODS'].copy()
print(ca1_foods.shape)

# Melt to long format
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in ca1_foods.columns if c.startswith('d_')]

ca1_foods_long = ca1_foods.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

print(ca1_foods_long.shape)

# Now merge in calendar
ca1_foods_long = ca1_foods_long.merge(calendar, on='d', how='left')
print(ca1_foods_long.shape)
ca1_foods_long.head()

(1437, 1947)
(2789217, 8)
(2789217, 21)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0


In [17]:
print(prices.shape)
prices.head()

(6841121, 4)


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [18]:
# now we have only foods data
# and we can see the prices table here
# lets join them

In [19]:
ca1_foods_long = ca1_foods_long.merge(
    prices,
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)
print(ca1_foods_long.shape)
ca1_foods_long.head()

(2789217, 22)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.00
1,FOODS_1_002_CA_1_evaluation,FOODS_1_002,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,7.88
2,FOODS_1_003_CA_1_evaluation,FOODS_1_003,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.88
3,FOODS_1_004_CA_1_evaluation,FOODS_1_004,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,NaN
4,FOODS_1_005_CA_1_evaluation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.94


In [20]:
print(ca1_foods_long.shape)

# How many rows have missing price?
print(ca1_foods_long['sell_price'].isna().sum())
print(ca1_foods_long['sell_price'].isna().mean())

(2789217, 22)
528976
0.18965035707153657


In [21]:
ca1_foods_long[ca1_foods_long['sell_price'].isna()]['sales'].describe()

count    528976.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: sales, dtype: float64

In [22]:
# For each item, find the first date where sell_price is NOT null (its launch date)
launch_dates = (
    ca1_foods_long[ca1_foods_long['sell_price'].notna()]
    .groupby('item_id')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'launch_date'})
)

print(launch_dates.shape)
launch_dates.head()

(1437, 2)


,item_id,launch_date
0,FOODS_1_001,2011-01-29
1,FOODS_1_002,2011-01-29
2,FOODS_1_003,2011-01-29
3,FOODS_1_004,2012-03-03
4,FOODS_1_005,2011-01-29


In [23]:
ca1_foods_long = ca1_foods_long.merge(launch_dates, on='item_id', how='left')

# Keep only rows on/after each item's own launch date
ca1_foods_trimmed = ca1_foods_long[ca1_foods_long['date'] >= ca1_foods_long['launch_date']].copy()

print(ca1_foods_long.shape)
print(ca1_foods_trimmed.shape)

(2789217, 23)
(2260241, 23)


In [24]:
# Check 1: should be 0 (or very near it) now
print(ca1_foods_trimmed['sell_price'].isna().sum())

# Check 2: pick an item that had missing prices, check its date range
sample_item = ca1_foods_long[ca1_foods_long['sell_price'].isna()]['item_id'].iloc[0]
print(sample_item)
print(ca1_foods_trimmed[ca1_foods_trimmed['item_id'] == sample_item]['date'].min())

0
FOODS_1_004
2012-03-03
